# Practical 2: 2D Spatial Convolution From Scratch Using NumPy
**Course:** CSE471 — Deep Learning for Computer Vision  
**Unit:** Unit II (Convolutional Neural Networks) | **CO:** CO2  
**Problem Statement Context:** SIH26131 — Manual Feature Extraction (Edge & Lesion Boundary Detection)  

---

## 1. Discrete 2D Convolution Mathematics
In image processing and deep learning, the discrete 2D spatial cross-correlation / convolution of an image $I$ with a kernel filter $K \in \mathbb{R}^{k_h 	imes k_w}$ at pixel coordinate $(i, j)$ is given by:

$$(I * K)(i, j) = \sum_{m=-k_h/2}^{k_h/2} \sum_{n=-k_w/2}^{k_w/2} I(i - m, j - n) \cdot K(m, n)$$

### Key Kernels:
- **Sobel Horizontal ($G_x$):** Detects vertical edges / vein boundaries
- **Sobel Vertical ($G_y$):** Detects horizontal edges
- **Laplacian ($
abla^2$):** Second-order derivative highlighting rapid intensity changes (lesion borders)
- **Gaussian Blur:** Low-pass filter for noise attenuation:
  $$G(x, y) = rac{1}{2\pi \sigma^2} e^{-rac{x^2+y^2}{2\sigma^2}}$$

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

## 2. 2D Convolution Algorithm (Pure NumPy Implementation)

In [ ]:
def convolve2d_scratch(image: np.ndarray, kernel: np.ndarray, stride: int = 1, padding: int = 0) -> np.ndarray:
    """
    Performs 2D discrete spatial convolution from first principles.
    """
    if padding > 0:
        image = np.pad(image, ((padding, padding), (padding, padding)), mode='constant', constant_values=0)

    img_h, img_w = image.shape
    k_h, k_w = kernel.shape

    out_h = (img_h - k_h) // stride + 1
    out_w = (img_w - k_w) // stride + 1
    output = np.zeros((out_h, out_w), dtype=np.float32)

    for i in range(out_h):
        for j in range(out_w):
            r_start = i * stride
            c_start = j * stride
            patch = image[r_start : r_start + k_h, c_start : c_start + k_w]
            output[i, j] = np.sum(patch * kernel)

    return output

print("[✓] convolve2d_scratch defined successfully.")

## 3. Define Agricultural Feature Extraction Filters

In [ ]:
# 1. Sobel Horizontal (Vertical Edges)
sobel_x = np.array([
    [-1, 0, 1],
    [-2, 0, 2],
    [-1, 0, 1]
], dtype=np.float32)

# 2. Sobel Vertical (Horizontal Edges)
sobel_y = np.array([
    [-1, -2, -1],
    [ 0,  0,  0],
    [ 1,  2,  1]
], dtype=np.float32)

# 3. Laplacian (Spot & Omnidirectional Edge Detector)
laplacian = np.array([
    [ 0,  1,  0],
    [ 1, -4,  1],
    [ 0,  1,  0]
], dtype=np.float32)

# 4. Sharpen Filter
sharpen = np.array([
    [ 0, -1,  0],
    [-1,  5, -1],
    [ 0, -1,  0]
], dtype=np.float32)

# 5. Gaussian Blur (3x3 Normalized)
gaussian_blur = (1.0 / 16.0) * np.array([
    [1, 2, 1],
    [2, 4, 2],
    [1, 2, 1]
], dtype=np.float32)

## 4. Run Convolutions on Leaf Image

In [ ]:
sample_path = REPO_ROOT / "data" / "samples" / "classification" / "lab_plantvillage" / "Tomato___Early_Blight" / "sample_000.jpg"
leaf_gray = np.array(Image.open(sample_path).convert("L"), dtype=np.float32) / 255.0

# Apply convolution with same padding (P=1)
f_sobel_x = convolve2d_scratch(leaf_gray, sobel_x, padding=1)
f_sobel_y = convolve2d_scratch(leaf_gray, sobel_y, padding=1)
sobel_magnitude = np.sqrt(f_sobel_x**2 + f_sobel_y**2)

f_laplace = convolve2d_scratch(leaf_gray, laplacian, padding=1)
f_sharpen = convolve2d_scratch(leaf_gray, sharpen, padding=1)
f_blur = convolve2d_scratch(leaf_gray, gaussian_blur, padding=1)

# Visualization
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes[0, 0].imshow(leaf_gray, cmap="gray")
axes[0, 0].set_title("Input Leaf (Grayscale)")
axes[0, 0].axis("off")

axes[0, 1].imshow(np.abs(f_sobel_x), cmap="magma")
axes[0, 1].set_title("Sobel X (Vertical Veins)")
axes[0, 1].axis("off")

axes[0, 2].imshow(np.abs(f_sobel_y), cmap="magma")
axes[0, 2].set_title("Sobel Y (Horizontal Edges)")
axes[0, 2].axis("off")

axes[1, 0].imshow(sobel_magnitude, cmap="inferno")
axes[1, 0].set_title("Sobel Gradient Magnitude")
axes[1, 0].axis("off")

axes[1, 1].imshow(np.abs(f_laplace), cmap="hot")
axes[1, 1].set_title("Laplacian (Lesion Boundaries)")
axes[1, 1].axis("off")

axes[1, 2].imshow(np.clip(f_sharpen, 0, 1), cmap="gray")
axes[1, 2].set_title("Sharpened Leaf Texture")
axes[1, 2].axis("off")

plt.tight_layout()
plt.show()

## 5. Practical Conclusion & Connection to CNNs
- Hand-crafted filters (Sobel, Laplacian) require manual parameter tuning and struggle with illumination changes in real-world fields.
- Deep CNNs replace these fixed filter matrices with **learnable weight tensors**, optimizing thousands of specialized filters directly via backpropagation.